## PRACTICA OBLIGATORIA: **Otros Modelos Supervisados**

* La práctica obligatoria de esta unidad consiste en resolver un modelado de clasificación, incluyendo KNN entre los posibles modelos, y aplicando balanceado.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo.
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso.

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, recall_score)
import warnings
warnings.filterwarnings('ignore')
print("Librerías cargadas correctamente")

### #1. El problema y los datos

Vamos a trabajar con el dataset 'Give me some credit'. El objetivo es predecir si una persona va a encontrarse en dificultades financieras en los dos próximos años.

### #1.1 Carga y descripción de variables

In [ ]:
df = pd.read_csv('data/credit_npo.csv')
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Descripción textual de las variables
descripcion = {
    'SeriousDlqin2yrs':                       'TARGET: Dificultades financieras en 2 años (0=No, 1=Sí)',
    'RevolvingUtilizationOfUnsecuredLines':    'Uso de crédito rotativo no garantizado (ratio 0-1)',
    'age':                                     'Edad en años',
    'NumberOfTime30-59DaysPastDueNotWorse':    'Nº veces con 30-59 días de retraso (no peor)',
    'DebtRatio':                               'Ratio deuda/ingresos mensuales',
    'MonthlyIncome':                           'Ingresos mensuales (€)',
    'NumberOfOpenCreditLinesAndLoans':         'Nº líneas de crédito y préstamos abiertos',
    'NumberOfTimes90DaysLate':                 'Nº veces con 90+ días de retraso',
    'NumberRealEstateLoansOrLines':            'Nº préstamos hipotecarios',
    'NumberOfTime60-89DaysPastDueNotWorse':    'Nº veces con 60-89 días de retraso (no peor)',
    'NumberOfDependents':                      'Nº dependientes a cargo'
}
for col, desc in descripcion.items():
    print(f"  {col:<45} -> {desc}")

In [ ]:
# Clasificación inicial: numéricas vs categóricas
print("Numéricas continuas:", ['RevolvingUtilizationOfUnsecuredLines','DebtRatio','MonthlyIncome'])
print("Numéricas discretas:", ['age','NumberOfTime30-59DaysPastDueNotWorse',
                               'NumberOfOpenCreditLinesAndLoans','NumberOfTimes90DaysLate',
                               'NumberRealEstateLoansOrLines','NumberOfTime60-89DaysPastDueNotWorse',
                               'NumberOfDependents'])
print("Binaria (target):    ['SeriousDlqin2yrs']")
print("\nNo hay variables categóricas de texto en este dataset.")

### #1.2 Tipo de problema y variable target

In [ ]:
print("Tipo de problema: CLASIFICACIÓN BINARIA")
print("Variable target: SeriousDlqin2yrs")
print("  0 = No experimenta dificultades financieras en 2 años")
print("  1 = Sí experimenta dificultades financieras en 2 años")

### #1.3 Distribución de frecuencias del target

In [ ]:
print("Distribución del target:")
print(df['SeriousDlqin2yrs'].value_counts())
print()
print("Distribución relativa:")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True).round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['SeriousDlqin2yrs'].value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue','coral'], edgecolor='black')
axes[0].set_title('Distribución target (absoluta)')
axes[0].set_xlabel('Dificultades financieras')
axes[0].tick_params(rotation=0)

df['SeriousDlqin2yrs'].value_counts(normalize=True).plot(kind='bar', ax=axes[1],
    color=['steelblue','coral'], edgecolor='black')
axes[1].set_title('Distribución target (porcentaje)')
axes[1].set_xlabel('Dificultades financieras')
axes[1].tick_params(rotation=0)
plt.tight_layout()
plt.show()

print('''
Comentario: El dataset está muy desbalanceado. Solo el ~6.9% de las personas experimentaron
dificultades financieras. Esto hace que el recall sea la métrica más relevante (objetivo de
negocio: detectar el mayor número posible de casos positivos).
Se aplicarán técnicas de balanceo (class_weight=balanced) para compensarlo.
''')

### Preprocesamiento

In [ ]:
# Imputación de valores faltantes
print("Valores faltantes antes:")
print(df.isnull().sum())

df['MonthlyIncome']    = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

print("\nValores faltantes después:", df.isnull().sum().sum())

### #2 Modelado

In [ ]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Positivos en train: {y_train.mean():.2%}")

# Escalado (necesario para KNN y LR)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# Definir CV estratificado
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = 'recall_macro'

# --- Modelo 1: KNN ---
knn = KNeighborsClassifier(n_neighbors=5)
knn_cv = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring=scoring)
print(f"KNN (k=5)         - Recall medio CV: {knn_cv.mean():.4f} ± {knn_cv.std():.4f}")

In [ ]:
# --- Modelo 2: Regresión Logística (con balanceo) ---
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_cv = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring=scoring)
print(f"Regresión Logística - Recall medio CV: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

In [ ]:
# --- Modelo 3: Random Forest (con balanceo) ---
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring=scoring)
print(f"Random Forest     - Recall medio CV: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

In [ ]:
# Comparación visual
resultados = {
    'KNN (k=5)': knn_cv.mean(),
    'Log. Regression': lr_cv.mean(),
    'Random Forest': rf_cv.mean()
}

print("\nComparación Recall Medio en CV:")
print("-" * 45)
for m, s in sorted(resultados.items(), key=lambda x: -x[1]):
    print(f"  {m:<20} Recall medio: {s:.4f}")

plt.figure(figsize=(7, 4))
plt.bar(resultados.keys(), resultados.values(), color=['steelblue','coral','mediumseagreen'], edgecolor='black')
plt.title('Recall medio en CV por modelo (con balanceo)')
plt.ylabel('Recall Macro')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Optimización del mejor modelo
print("Optimizando el mejor modelo (Random Forest)...")

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 5],
    'class_weight': ['balanced']
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    verbose=0
)
grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Recall medio en CV (optimizado): {grid_rf.best_score_:.4f}")

In [ ]:
# Evaluación sobre test
best_rf = grid_rf.best_estimator_
y_pred  = best_rf.predict(X_test)

print("=" * 65)
print("EVALUACIÓN FINAL - Random Forest Optimizado (TEST SET)")
print("=" * 65)
print(classification_report(y_test, y_pred, target_names=['Sin dificultades','Con dificultades']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Sin dificultades','Con dificultades']).plot(cmap='Blues', ax=ax)
ax.set_title('Matriz de Confusión - Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de errores
y_test_arr = np.array(y_test)
y_pred_arr = np.array(y_pred)

fp = np.where((y_pred_arr == 1) & (y_test_arr == 0))[0]
fn = np.where((y_pred_arr == 0) & (y_test_arr == 1))[0]

print(f"Falsos Positivos (FP): {len(fp)} - personas sin dificultades clasificadas como con dificultades")
print(f"Falsos Negativos (FN): {len(fn)} - personas con dificultades NO detectadas")
print()

# Características de los falsos negativos vs verdaderos positivos
tp = np.where((y_pred_arr == 1) & (y_test_arr == 1))[0]
X_test_df = pd.DataFrame(X_test_sc, columns=X.columns)

print("Comparativa FN vs TP en features clave:")
key_features = ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTimes90DaysLate', 'DebtRatio']
for feat in key_features:
    fn_mean = X_test_df.iloc[fn][feat].mean() if len(fn) > 0 else 0
    tp_mean = X_test_df.iloc[tp][feat].mean() if len(tp) > 0 else 0
    print(f"  {feat:<45} FN: {fn_mean:+.3f} | TP: {tp_mean:+.3f}  (estandarizado)")

print('''
Interpretación:
- Los FN suelen tener perfiles menos extremos en indicadores de riesgo.
- Mejora potencial: incluir interacciones entre variables de retraso de pago,
  o ajustar el umbral de clasificación para reducir FN a costa de más FP.
''')